# Convolutional Neural Networks

## Project: Write an Algorithm for Landmark Classification

### Install Prerequisites

To run the app in the notebook environment, you must first install the required packages by executing the two cells below. **Make sure to restart the kernel after running each cell.**

> Note: Restarting the kernel ensures that all installed dependencies are properly loaded into the environment.

In [1]:
# Please restart the notebook kernel after running this cell.
!pip install -r requirements.txt | grep -v "already satisfied"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 30.5 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py): finished with status 'error'
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [5]:
!pip install gradio


### A Simple App

In this notebook we build a very simple app that uses our exported model.


### Test Your App
Go to a search engine for images (like Google Images) and search for images of some of the landmarks, like the Eiffel Tower, the Golden Gate Bridge, Machu Picchu and so on.

The app will show the top 5 classes that the model think are most relevant for the picture you have uploaded

In [8]:
import gradio as gr
import torch
from torch import nn
from torchvision import models, transforms
from PIL import Image
import json

# Recreate the model architecture exactly as in training
def get_model_transfer_learning(model_name="resnet18", n_classes=50):
    model_transfer = getattr(models, model_name)(pretrained=True)
    for param in model_transfer.parameters():
        param.requires_grad = False
    num_ftrs = model_transfer.fc.in_features
    model_transfer.fc = nn.Linear(num_ftrs, n_classes)
    return model_transfer

# Initialize the same architecture
model = get_model_transfer_learning("resnet18", n_classes=50)

# Load trained weights
model.load_state_dict(torch.load("model_transfer.pt", map_location='cpu'))
model.eval()

# Define preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Define your class names
with open("class_names.json", "r") as f:
    class_names = json.load(f)

# Prediction function
def classify_landmark(img):
    try:
        timg = transform(img).unsqueeze(0)
        with torch.no_grad():
            outputs = model(timg)
            probs = torch.nn.functional.softmax(outputs, dim=1)[0]
            top5 = torch.topk(probs, 5)
            results = {}
            for idx, p in zip(top5.indices, top5.values):
                class_name = class_names[idx] if idx < len(class_names) else f"Class {idx.item()}"
                results[class_name] = round(float(p), 3)
        return results
    except Exception as e:
        print("⚠️ Error during inference:", e)
        return {"Error": str(e)}

# Gradio UI
demo = gr.Interface(
    fn=classify_landmark,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=5),
    title="🏛️ Landmark Classifier",
    description="Upload an image of a landmark and the model will predict the most likely site."
)

demo.launch(share=True)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/tmp/ipython-input-361809848.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more detai

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://99777282ae97493373.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
